In [39]:
import os
import warnings
import tqdm
import pandas as pd

import socceraction.spadl as spadl
import socceraction.vaep.formula as vaepformula

warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

In [40]:
datafolder = "../data-generated"
spadl_h5 = os.path.join(datafolder, "spadl-statsbomb.h5")
predictions_h5 = os.path.join(datafolder, "predictions.h5")

In [41]:
with pd.HDFStore(spadl_h5) as spadlstore:
    games = (
        spadlstore["games"]
        .merge(spadlstore["competitions"], how='left')
        .merge(spadlstore["teams"].add_prefix('home_'), how='left')
        .merge(spadlstore["teams"].add_prefix('away_'), how='left'))
    players = spadlstore["players"]
    teams = spadlstore["teams"]
print("nb of games:", len(games))

nb of games: 751


In [42]:
A = []
for game in tqdm.tqdm(list(games.itertuples()), desc="Rating actions"):
    actions = pd.read_hdf(spadl_h5, f"actions/game_{game.game_id}")
    actions = (
        spadl.add_names(actions)
        .merge(players, how="left")
        .merge(teams, how="left")
        .sort_values(["game_id", "period_id", "action_id"])
        .reset_index(drop=True)
    )
    preds = pd.read_hdf(predictions_h5, f"game_{game.game_id}")
    values = vaepformula.value(actions, preds.scores, preds.concedes)
    A.append(pd.concat([actions, preds, values], axis=1))
A = pd.concat(A).sort_values(["game_id", "period_id", "time_seconds"]).reset_index(drop=True)
A.columns

Rating actions: 100%|██████████| 751/751 [00:55<00:00, 13.59it/s]


Index(['game_id', 'original_event_id', 'period_id', 'time_seconds', 'team_id',
       'player_id', 'start_x', 'start_y', 'end_x', 'end_y', 'type_id',
       'result_id', 'bodypart_id', 'action_id', 'type_name', 'result_name',
       'bodypart_name', 'player_name', 'nickname', 'team_name', 'scores',
       'concedes', 'offensive_value', 'defensive_value', 'vaep_value'],
      dtype='object')

In [43]:
# Load all_fouls_advanced.pkl and testing_fouls_advanced.pkl
all_fouls_advanced = pd.read_pickle("all_fouls_advanced.pkl")
testing_fouls_advanced = pd.read_pickle("testing_fouls_advanced.pkl")

In [44]:
import numpy as np

all_fouls_advanced['vaep_value_offensive'] = np.nan
testing_fouls_advanced['vaep_value_offensive'] = np.nan


# Make original_event_id column in A dataframe as it's index
A = A.set_index('original_event_id')

# all_fouls_advanced['id'] has values from 'original_event_id' column in A dataframe. Fill the values of 'vaep_value_offensive' column in all_fouls_advanced dataframe with the values of 'offensive_value' column in A dataframe
for i in range(len(all_fouls_advanced)):
    all_fouls_advanced['vaep_value_offensive'][i] = A['offensive_value'][all_fouls_advanced['id'][i]]

for i in range(len(testing_fouls_advanced)):
    # if testing_fouls_advanced['id'][i] in A.index, then fill the value of 'vaep_value_offensive' column in testing_fouls_advanced dataframe with the value of 'offensive_value' column in A dataframe
    if testing_fouls_advanced['id'][i] in A.index:
        testing_fouls_advanced['vaep_value_offensive'][i] = A['offensive_value'][testing_fouls_advanced['id'][i]]

/tmp/ipykernel_30726/1484256975.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_fouls_advanced['vaep_value_offensive'][i] = A['offensive_value'][all_fouls_advanced['id'][i]]
/tmp/ipykernel_30726/1484256975.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  testing_fouls_advanced['vaep_value_offensive'][i] = A['offensive_value'][testing_fouls_advanced['id'][i]]


In [45]:
# Remove the rows in all_fouls_advanced and testing_fouls_advanced where vaep_value_offensive is NaN
all_fouls_advanced = all_fouls_advanced.dropna(subset=['vaep_value_offensive'])
testing_fouls_advanced = testing_fouls_advanced.dropna(subset=['vaep_value_offensive'])

In [46]:
testing_fouls_advanced['vaep_value_offensive'].isnull().sum()

0

In [47]:
len(all_fouls_advanced)

6807

In [48]:
len(testing_fouls_advanced)

947

In [49]:
# divide all_fouls_advanced vaept_value_offensive column into logical range and print their count
all_fouls_advanced['vaep_value_offensive'].value_counts(bins=10)

# divide testing_fouls_advanced vaept_value_offensive column into logical range and print their count
testing_fouls_advanced['vaep_value_offensive'].value_counts(bins=10)

(-0.00717, 0.0175]     829
(-0.0319, -0.00717]     83
(-0.0565, -0.0319]      19
(-0.106, -0.0812]        6
(-0.131, -0.106]         4
(-0.0812, -0.0565]       4
(-0.231, -0.205]         1
(-0.155, -0.131]         1
(-0.205, -0.18]          0
(-0.18, -0.155]          0
Name: vaep_value_offensive, dtype: int64

In [62]:
import numpy as np

type_dict = {'Handball': 0, 'Dangerous Play': 1, 'Foul Out': 2, 'Dive': 3, '6 Seconds': 4, 'Backpass Pick': 5}

card_dict = {'Yellow Card': 0, 'Second Yellow': 0, 'Red Card': 1}

def create_features(team_fouls_df):
    all_features = []
    all_labels = []

    for _, row in team_fouls_df.iterrows():
        # select features only where foul_committed_type is not in type_dict keys
        if row['foul_committed_type'] in type_dict.keys():
            continue
        
        team_making_foul = row['team']

        scoreline = row['scoreline_till_now']
        team_not_making_foul = [team for team in scoreline.keys() if team != team_making_foul][0]
        
        score_difference_till_now = scoreline[team_not_making_foul] - scoreline[team_making_foul]

        features = [row['minute'], score_difference_till_now, row['distance_to_goal'], row['angle_to_goal'], row['foul_count_player_till_now'], row['foul_count_team_till_now'], row['vaep_value_offensive'], row['id']]

        all_features.append(features)
        all_labels.append([card_dict.get(row['foul_committed_card'], 2)])
    
    return all_features, all_labels

In [63]:
import pickle
from tqdm import tqdm

all_the_features = []
all_the_labels = []

# load match_ids_list from file using pickle
with open('match_ids_list.pkl', 'rb') as f:
    match_ids_list = pickle.load(f)

# load match_ids_list_test from file using pickle
with open('match_ids_list_test.pkl', 'rb') as f:
    match_ids_list_test = pickle.load(f)

# use tqdm to iterate match_ids_list
for match_id in tqdm(match_ids_list):
    # get all fouls in the match
    match_fouls_df = all_fouls_advanced[all_fouls_advanced['match_id'] == match_id]

    feature, label = create_features(match_fouls_df)

    # append features and labels to all_the_features and all_the_labels
    all_the_features.extend(feature)
    all_the_labels.extend(label)

# save all_the_features and all_the_labels to files using pickle
with open('all_the_features.pkl', 'wb') as f:
    pickle.dump(all_the_features, f)

with open('all_the_labels.pkl', 'wb') as f:
    pickle.dump(all_the_labels, f)

100%|██████████| 623/623 [00:00<00:00, 820.49it/s]


In [64]:
all_the_features_test= []
all_the_labels_test = []

# use tqdm to iterate match_ids_list
for match_id in tqdm(match_ids_list_test):
    # get all fouls in the match
    match_fouls_df = testing_fouls_advanced[testing_fouls_advanced['match_id'] == match_id]

    feature, label = create_features(match_fouls_df)

    # append features and labels to all_the_features and all_the_labels
    all_the_features_test.extend(feature)
    all_the_labels_test.extend(label)

# save all_the_features and all_the_labels to files using pickle
with open('all_the_features_test.pkl', 'wb') as f:
    pickle.dump(all_the_features_test, f)

with open('all_the_labels_test.pkl', 'wb') as f:
    pickle.dump(all_the_labels_test, f)

100%|██████████| 128/128 [00:00<00:00, 1076.18it/s]


In [65]:
print (len(all_the_features))

# first 10 items in all_the_features and all_the_labels
print (all_the_features[:100])
print (all_the_labels[:100])

# print count of unique labels
print (np.unique(all_the_labels, return_counts=True))

5572
[[3, -1, 74.90367147209808, -0.2981063128391762, 1, 1, 0.0005195345729589462, 'a40924e7-b5ca-4f23-9025-ac535d63dab8'], [8, -1, 82.67823171790747, 0.3557978590960649, 1, 2, 8.534954395145178e-05, '24f49483-d437-40c2-bf5f-c6b3e5456d48'], [54, -1, 29.04134983088768, 1.2773606430795648, 1, 4, 4.988000728189945e-05, '681f5c57-ed69-4bae-82e7-6d97233fb907'], [74, -1, 102.00705857929637, -0.011764163149758776, 1, 5, -0.03393703559413552, '7ea7ec5e-388e-4b20-8ab5-d2359cb92a65'], [55, -1, 77.53192890674138, -0.41979583169048146, 1, 2, 0.00045737839536741376, '6238286d-a363-4ed3-abab-f322e0571e2d'], [81, -1, 81.17745992577004, 0.41459707844526655, 1, 4, -0.0010884791845455766, '928a15d8-a10e-411c-953d-301663f2fada'], [92, -1, 84.38009243891595, -0.3787458155510841, 2, 5, 4.4264597818255424e-05, 'd32eb3eb-ac10-4d1f-a2d6-1f7d1b1f035c'], [10, 0, 43.18564576337837, -0.09275632018814374, 1, 1, 0.0007246378809213638, 'caaeb0e8-e589-4586-bf9c-56f0e75d1adf'], [11, 0, 70.61161377563892, -0.2140606835

In [66]:
print (len(all_the_features_test))

# first 10 items in all_the_features and all_the_labels
print (all_the_features_test[:100])
print (all_the_labels_test[:100])

# print count of unique labels
print (np.unique(all_the_labels_test, return_counts=True))

757
[[14, 0, 37.96840792026972, 1.2823667682765016, 1, 1, -0.0003927641664631665, '9221da02-9c87-4f98-a8ea-de9994ad562b'], [23, 1, 65.01115288933123, 0.058484844490939804, 1, 1, -0.0012224323581904173, 'f2bd21f6-87c3-44c0-bb37-04e996db413e'], [33, 0, 69.64165420206503, -0.3063112726462804, 1, 2, -0.0010379071172792464, '8e51baec-6948-4285-86a1-73197126e265'], [46, 0, 52.10614167255142, -0.26406386883592514, 1, 2, -0.000850418204208836, 'b711f529-182e-4ae3-8c9a-189b7697b2ea'], [55, 1, 42.569942447694245, 0.5541172736434499, 1, 3, -0.007319043594179675, 'ec46c71c-af4c-4d99-a854-e4ffc30460d6'], [80, 1, 98.5211652387445, 0.09249778257668083, 1, 5, -0.0003770081093534827, '142138e3-20c9-46b2-80cd-db92a583bc81'], [86, 1, 64.82352967865913, 0.061745215024700977, 3, 6, -0.00014433148317039013, '3d99f9db-1f6c-45e1-b604-73f118e0bb16'], [98, -1, 33.64342432036311, 0.9415629774702307, 1, 3, -0.00045858632074669003, '4626dd9d-ced6-4b46-9fb6-01a9b74ba80a'], [99, 1, 31.49857139617605, -0.704494064242